# 03 — Physics proxies and summary statistics
1. Do the two physics proxies (ln α̂, ln β̂_jac) separate each fault category under realistic variation?
   Cross-sensitivity; the legacy UA_eff_proxy confound; robustness of the summaries.
2. **Summary-set selection** (S0 = 29 System I summaries; S1m = production; S1m_no_legacy).

**PRE-REGISTER (fill and date before running):** System I nb03 rule (`rules.summary_rule`), default S1m,
3 seeds × 20k from one bank, efficiency = mean posterior sd / Bayesian CRB of the same set at Sc1–Sc4.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

## 1. Proxy separation (threshold rule) and cross-sensitivity

In [ ]:
L = da.load_npz(DATA / "L_prod.npz")
ia, ib = fe.NAMES["S1m"].index("ln_alpha_hat"), fe.NAMES["S1m"].index("ln_beta_hat_jac")
rows = []
for k in scn.CLOSED:
    m = L["sc_name"] == k; s = scn.SCENARIOS[k]
    ah, bh = np.exp(L["x"][m, ia]), np.exp(L["x"][m, ib])
    ok = ((ah < scn.THRESHOLD) == (s.alpha < scn.THRESHOLD)) & ((bh < scn.THRESHOLD) == (s.beta < scn.THRESHOLD))
    rows.append(dict(scenario=k, alpha=s.alpha, beta=s.beta, proxy_rule_acc=ok.mean(), alpha_hat=ah.mean(),
                     alpha_hat_sd=ah.std(), beta_hat=bh.mean(), beta_hat_sd=bh.std()))
prox = pd.DataFrame(rows).set_index("scenario"); display(prox.round(4)); prox.to_csv(R / "03_proxy_rule.csv")
A = [0.6, 0.8, 1.0, 1.15]; M = {}
iu = fe.NAMES["S1m"].index("UA_eff_proxy")
for a in A:
    for b in A:
        d = da.make_windows(np.tile([a, b], (300, 1)), 3100 + int(100 * a + b * 10), conditions="production")
        M[(a, b)] = (np.exp(d["x"][:, ia]).mean(), np.exp(d["x"][:, ib]).mean(), d["x"][:, iu].mean())
cross = dict(
    d_alphahat_d_beta=[(M[(a, A[j + 1])][0] - M[(a, A[j])][0]) / (A[j + 1] - A[j]) for a in A for j in range(3)],
    d_betahat_d_alpha=[(M[(A[i + 1], b)][1] - M[(A[i], b)][1]) / (A[i + 1] - A[i]) for b in A for i in range(3)],
    UAproxy_alpha_sens=[(M[(A[i + 1], b)][2] - M[(A[i], b)][2]) / (A[i + 1] - A[i]) for b in A for i in range(3)],
    UAproxy_beta_sens=[(M[(a, A[j + 1])][2] - M[(a, A[j])][2]) / (A[j + 1] - A[j]) for a in A for j in range(3)])
display(pd.DataFrame({k: [np.min(v), np.max(v)] for k, v in cross.items()}, index=["min", "max"]).round(4))
da.save_json(R / "03_cross_sensitivity.json", {k: [float(x) for x in v] for k, v in cross.items()})

## 2. Summary robustness: warm start, constant features, clipping

In [ ]:
x2 = da.make_windows(np.tile([1.0, 1.0], (800, 1)), 3300, protocol="P2")["x"]
x1 = da.make_windows(np.tile([1.0, 1.0], (800, 1)), 3300, protocol="P1b")["x"]
sd = x2.std(0); shift = np.where(sd > 0, (x1.mean(0) - x2.mean(0)) / np.where(sd > 0, sd, 1), 0)
rob = pd.DataFrame({"feature": fe.NAMES["S1m"], "shift_P1b_vs_P2_sd": shift, "sd": sd}).set_index("feature")
display(rob.sort_values("shift_P1b_vs_P2_sd", key=np.abs, ascending=False).head(8).round(3))
print("constant features at nominal:", list(rob.index[rob.sd < 1e-12]))
rob.to_csv(R / "03_summary_robustness.csv")

## 3. Summary-set selection (training)

In [ ]:
from system_iii import inference as inf
SETS = {"S0": fe.NAMES["S0"], "S1m": fe.NAMES["S1m"], "S1m_no_legacy": fe.NAMES["S1m_no_legacy"]}
col = {k: [fe.NAMES["S1m"].index(n) for n in v] for k, v in SETS.items()}
bank_f = DATA / "03_bank_S1m_60k.npz"
if not bank_f.exists():
    da.save_npz(bank_f, da.make_windows(sim.prior_draws(60_000, 3400), 3400, conditions="production"))
bank = da.load_npz(bank_f)
test = da.test_set(3499, conditions="production")
rows = []
for sname, idx in col.items():
    for s in range(3):
        f = R / "03" / f"post_{sname}_s{s}.pkl"; f.parent.mkdir(exist_ok=True)
        if f.exists():
            post = __import__("pickle").load(open(f, "rb"))
        else:
            sl = slice(s * 20_000, (s + 1) * 20_000)
            post, meta = inf.train_npe(bank["theta"][sl], bank["x"][sl][:, idx], seed=s)
            __import__("pickle").dump(post, open(f, "wb"))
        sub = dict(test, x_sbc=test["x_sbc"][:, idx], x_sc=test["x_sc"][:, idx])
        row, _ = inf.evaluate(post, sub, seed=s)
        rows.append({"set": sname, "seed": s, "n_features": len(idx), **row})
met = pd.DataFrame(rows); met.to_csv(R / "03_summary_sets_metrics.csv", index=False)

In [ ]:
crb = {s: {pt: info.bayes_crb(scn.SCENARIOS[n].alpha, scn.SCENARIOS[n].beta, feature_set=s, n=1000)
           for pt, n in zip(["Sc1", "Sc2", "Sc3", "Sc4"], scn.GATED)} for s in SETS}
def eff_row(r):
    return np.mean([r[f"sd{p}_{pt}"] / crb[r["set"]][pt]["sd_" + ("alpha" if p == "a" else "beta")]
                    for pt in ["Sc1", "Sc2", "Sc3", "Sc4"] for p in ("a", "b")])
met["eff"] = met.apply(eff_row, axis=1)
se = lambda v: v.std(ddof=1) / np.sqrt(len(v))
agg = met.groupby("set").agg(n_features=("n_features", "first"), ks_a=("ks_p_a_interior", "median"), ks_b=("ks_p_b_interior", "median"),
                             tarp=("tarp_ks_p", "median"), rmse=("rmse_norm", "mean"), rmse_se=("rmse_norm", se),
                             eff=("eff", "mean"), eff_se=("eff", se))
agg["calib_ok"] = (agg.ks_a >= 0.01) & (agg.ks_b >= 0.01) & (agg.tarp >= 0.01)
dec = rules.summary_rule(agg, default="S1m"); display(agg.round(4)); print(dec)
da.save_json(R / "03_summary_decision.json", dict(decision=dec, table=agg.reset_index().to_dict("records")))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).